In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import glob
import pandas as pd
import numpy as np

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
CLEAN_DIR = "/content/drive/MyDrive/Economic_Times/Dataset/CSE-CIC-IDS2018/CLEANED"

TRAIN_FILE = "/content/drive/MyDrive/Economic_Times/CICIDS2018_train_ready.csv"


clean_files = sorted(
    glob.glob(CLEAN_DIR+"/*.csv")
)


print("Total chunks:", len(clean_files))


samples = []

ROWS_PER_CHUNK = 10000


for f in clean_files:

    print("Processing:", os.path.basename(f))

    df = pd.read_csv(
        f,
        low_memory=False
    )


    # take random samples
    if len(df) > ROWS_PER_CHUNK:
        df = df.sample(
            ROWS_PER_CHUNK,
            random_state=42
        )


    samples.append(df)



# combine only sampled data
train_data = pd.concat(
    samples,
    ignore_index=True
)


print("Final shape:", train_data.shape)


# save permanently
train_data.to_csv(
    TRAIN_FILE,
    index=False
)


print("Saved training file")

Total chunks: 168
Processing: chunk_0.csv
Processing: chunk_1.csv
Processing: chunk_10.csv
Processing: chunk_100.csv
Processing: chunk_101.csv
Processing: chunk_102.csv
Processing: chunk_103.csv
Processing: chunk_104.csv
Processing: chunk_105.csv
Processing: chunk_106.csv
Processing: chunk_107.csv
Processing: chunk_108.csv
Processing: chunk_109.csv
Processing: chunk_11.csv
Processing: chunk_110.csv
Processing: chunk_111.csv
Processing: chunk_112.csv
Processing: chunk_113.csv
Processing: chunk_114.csv
Processing: chunk_115.csv
Processing: chunk_116.csv
Processing: chunk_117.csv
Processing: chunk_118.csv
Processing: chunk_119.csv
Processing: chunk_12.csv
Processing: chunk_120.csv
Processing: chunk_121.csv
Processing: chunk_122.csv
Processing: chunk_123.csv
Processing: chunk_124.csv
Processing: chunk_125.csv
Processing: chunk_126.csv
Processing: chunk_127.csv
Processing: chunk_128.csv
Processing: chunk_129.csv
Processing: chunk_13.csv
Processing: chunk_130.csv
Processing: chunk_131.csv
Pr

In [ ]:
import gc

del samples
del df

gc.collect()

0

In [ ]:
data = pd.read_csv(
    TRAIN_FILE,
    low_memory=False
)

print(data.shape)

(1656800, 79)


In [ ]:
data.insert(
    0,
    "Unique_ID",
    range(
        1,
        len(data)+1
    )
)

print(data.head())

   Unique_ID Dst Port Protocol Flow Duration Tot Fwd Pkts Tot Bwd Pkts  \
0          1     3389        6        234438            3            1   
1          2     8080        6          9953            3            4   
2          3     8080        6         11279            3            4   
3          4     3389        6       1393073            8            7   
4          5     8080        6         10792            3            4   

  TotLen Fwd Pkts TotLen Bwd Pkts Fwd Pkt Len Max Fwd Pkt Len Min  ...  \
0               0             0.0               0               0  ...   
1             326           129.0             326               0  ...   
2             326           129.0             326               0  ...   
3            1132          1581.0             661               0  ...   
4             326           129.0             326               0  ...   

  Fwd Seg Size Min Active Mean Active Std Active Max Active Min Idle Mean  \
0               20         0.0   

In [ ]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

data["Label"] = encoder.fit_transform(
    data["Label"]
)

print(encoder.classes_)

['Benign' 'Bot' 'Brute Force -Web' 'Brute Force -XSS' 'DDOS attack-HOIC'
 'DDOS attack-LOIC-UDP' 'DDoS attacks-LOIC-HTTP' 'DoS attacks-GoldenEye'
 'DoS attacks-Hulk' 'DoS attacks-SlowHTTPTest' 'DoS attacks-Slowloris'
 'FTP-BruteForce' 'Infilteration' 'Label' 'SQL Injection' 'SSH-Bruteforce']


In [ ]:
X = data.drop(
    columns=[
        "Label",
        "Unique_ID"
    ]
)

y = data["Label"]

print(X.shape)
print(y.shape)

(1656800, 78)
(1656800,)


In [ ]:
from sklearn.preprocessing import StandardScaler
import numpy as np

# Replace infinity values
X_train = X_train.replace(
    [np.inf, -np.inf],
    np.nan
)

X_test = X_test.replace(
    [np.inf, -np.inf],
    np.nan
)

# Fill missing values
X_train = X_train.fillna(0)
X_test = X_test.fillna(0)


scaler = StandardScaler()

X_train = scaler.fit_transform(
    X_train
)

X_test = scaler.transform(
    X_test
)

print("Scaling complete")

Scaling complete


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(1325440, 78)
(331360, 78)


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

In [ ]:
X_train = X_train.reshape(
    X_train.shape[0],
    X_train.shape[1],
    1
)

X_test = X_test.reshape(
    X_test.shape[0],
    X_test.shape[1],
    1
)

print(X_train.shape)

(1325440, 78, 1)


In [ ]:
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv1D,
    MaxPooling1D,
    BatchNormalization,
    Flatten,
    Dense,
    Dropout
)


num_classes = len(
    np.unique(y_train)
)


model = Sequential()


model.add(
    Conv1D(
        128,
        3,
        activation="relu",
        input_shape=(
            X_train.shape[1],
            1
        )
    )
)

model.add(BatchNormalization())
model.add(MaxPooling1D(2))


model.add(
    Conv1D(
        256,
        3,
        activation="relu"
    )
)

model.add(BatchNormalization())
model.add(MaxPooling1D(2))


model.add(
    Conv1D(
        512,
        3,
        activation="relu"
    )
)

model.add(BatchNormalization())


model.add(
    Flatten()
)


model.add(
    Dense(
        4096,
        activation="relu"
    )
)

model.add(
    Dropout(0.5)
)


model.add(
    Dense(
        2048,
        activation="relu"
    )
)

model.add(
    Dropout(0.5)
)


model.add(
    Dense(
        1024,
        activation="relu"
    )
)

model.add(
    Dropout(0.5)
)


model.add(
    Dense(
        num_classes,
        activation="softmax"
    )
)


model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_6 (Conv1D)               │ (None, 76, 128)        │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 76, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_4 (MaxPooling1D)  │ (None, 38, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_7 (Conv1D)               │ (None, 36, 256)        │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 36, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_5 (MaxPooling1D)  │ (None, 18, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_8 (Conv1D)               │ (None, 16, 512)        │       393,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 16, 512)        │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 4096)           │    33,558,528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 2048)           │     8,390,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1024)           │     2,098,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 16)             │        16,400 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 44,560,144 (169.98 MB)

 Trainable params: 44,558,352 (169.98 MB)

 Non-trainable params: 1,792 (7.00 KB)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=4096,
    validation_split=0.2
)

Epoch 1/10
259/259 ━━━━━━━━━━━━━━━━━━━━ 181s 566ms/step - accuracy: 0.9731 - loss: 0.2415 - val_accuracy: 0.8436 - val_loss: 1.6001
Epoch 2/10
259/259 ━━━━━━━━━━━━━━━━━━━━ 113s 437ms/step - accuracy: 0.9858 - loss: 0.0741 - val_accuracy: 0.8835 - val_loss: 0.6512
Epoch 3/10
259/259 ━━━━━━━━━━━━━━━━━━━━ 115s 445ms/step - accuracy: 0.9866 - loss: 0.0658 - val_accuracy: 0.9872 - val_loss: 0.0699
Epoch 4/10
259/259 ━━━━━━━━━━━━━━━━━━━━ 116s 449ms/step - accuracy: 0.9869 - loss: 0.0615 - val_accuracy: 0.9872 - val_loss: 0.0639
Epoch 5/10
259/259 ━━━━━━━━━━━━━━━━━━━━ 117s 453ms/step - accuracy: 0.9871 - loss: 0.0593 - val_accuracy: 0.9871 - val_loss: 0.0591
Epoch 6/10
259/259 ━━━━━━━━━━━━━━━━━━━━ 117s 453ms/step - accuracy: 0.9873 - loss: 0.0583 - val_accuracy: 0.9870 - val_loss: 0.0606
Epoch 7/10
259/259 ━━━━━━━━━━━━━━━━━━━━ 117s 451ms/step - accuracy: 0.9874 - loss: 0.0576 - val_accuracy: 0.9878 - val_loss: 0.0568
Epoch 8/10
259/259 ━━━━━━━━━━━━━━━━━━━━ 116s 449ms/step - accuracy: 0.9874 -

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)


pred = model.predict(
    X_test
)


pred = np.argmax(
    pred,
    axis=1
)


print(
    "Accuracy:",
    accuracy_score(
        y_test,
        pred
    )
)


print(
    "Precision:",
    precision_score(
        y_test,
        pred,
        average="weighted"
    )
)


print(
    "Recall:",
    recall_score(
        y_test,
        pred,
        average="weighted"
    )
)


print(
    "F1 Score:",
    f1_score(
        y_test,
        pred,
        average="weighted"
    )
)


print(
    classification_report(
        y_test,
        pred
    )
)

10355/10355 ━━━━━━━━━━━━━━━━━━━━ 26s 2ms/step
Accuracy: 0.9878621438918397
Precision: 0.9848203043591881
Recall: 0.9878621438918397
F1 Score: 0.9845965542483747
              precision    recall  f1-score   support

           0       0.99      1.00      0.99    286959
           1       1.00      1.00      1.00      4699
           2       1.00      0.29      0.44        14
           3       1.00      0.50      0.67         4
           4       1.00      1.00      1.00      9530
           5       0.80      1.00      0.89        47
           6       0.99      0.99      0.99     11557
           7       0.98      0.99      0.99       862
           8       1.00      1.00      1.00      8238
           9       0.00      0.00      0.00        21
          10       0.84      0.99      0.91       210
          11       0.43      1.00      0.60        18
          12       0.65      0.17      0.27      4031
          14       0.00      0.00      0.00         1
          15       1.00     

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

In [ ]:
MODEL_PATH = "/content/drive/MyDrive/Economic_Times/CICIDS2018_CNN_Model.keras"

model.save(
    MODEL_PATH
)

print(
    "Model saved:",
    MODEL_PATH
)

Model saved: /content/drive/MyDrive/Economic_Times/CICIDS2018_CNN_Model.keras
